# CUDA/T4 V-14 0.5x core: non-value preparation

This notebook prepares the independent 0.5x V-14 core pass. It verifies the already-qualified CUDA/T4 controls, performs one bounded 0.5x timing measurement with records discarded, and emits a capped 0.5x batch plan.

It contains no `--run-batch` invocation and cannot retain V-14 scientific records, minima, ties, labels, rates, comparisons, or conclusions. A later owner authorization bound to the emitted 0.5x plan would be required before production.

In [ ]:
import shutil
import subprocess

assert shutil.which('nvidia-smi'), 'No GPU backend: select a T4 runtime, reconnect, then rerun.'
assert shutil.which('nvcc'), 'nvcc is unavailable: this is not the required CUDA/T4 runtime.'
gpu = subprocess.check_output(['nvidia-smi', '--query-gpu=name,driver_version', '--format=csv,noheader'], text=True).strip()
assert gpu.startswith('Tesla T4,'), f'Required Tesla T4, got: {gpu}'
print('GPU:', gpu)
print(subprocess.check_output(['nvcc', '--version'], text=True))

In [ ]:
from google.colab import files
from pathlib import Path
import hashlib
import tarfile

EXPECTED_CAPSULE_SHA256 = 'ef0681f50165615cf9a862778f80ce65b58b2b643a7fe4d2612a16dc59b71337'
uploaded = files.upload()  # Select g6_cuda_t4_v14_core_final_2x_capsule.tar.gz
assert len(uploaded) == 1, 'Upload exactly one capsule.'
archive = Path(next(iter(uploaded)))
actual = hashlib.sha256(archive.read_bytes()).hexdigest()
print(f'{archive.name}: {actual}')
assert actual == EXPECTED_CAPSULE_SHA256, 'Wrong capsule; do not continue.'
with tarfile.open(archive, 'r:gz') as bundle:
    bundle.extractall('/content', filter='data')
ROOT = Path('/content/project')
CONTROL_INPUT = Path('/content/g6_v14_2x_core_control')
assert (ROOT / 'research/g6/run_cuda_v14_core_batches.py').is_file() and CONTROL_INPUT.is_dir()
print('Capsule extracted:', ROOT)

In [ ]:
from google.colab import drive
import json
import os
import sys

sys.dont_write_bytecode = True
drive.mount('/content/gdrive')
PREP_LABEL = 'g6_cuda_t4_v14_core_0_5x_preparation_20260921'
OUT = Path('/content/gdrive/MyDrive') / PREP_LABEL
OUT.mkdir(parents=True, exist_ok=True)

EXPECTED_FILES = {
    'unbalanced_qualification.json': 'e3a95d24ea257da47c6ab8654ce137bc6ab1b35e19ac187cacd5ad2bda40b890',
    'slack_qualification.json': 'd60a90413fa7ed2ce83ce2f175ea7bddf7d9ad672ca1e7a9dbf5e93f3946278b',
    'v14_core_validation.json': 'ec959f85e665bad187f23b21ce6abea7becce2994e6889c12e9dc269a27b13f0',
}
def digest(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()
for name, wanted in EXPECTED_FILES.items():
    assert digest(CONTROL_INPUT / name) == wanted, name
sys.path.insert(0, str(ROOT))
from research.g6.cuda_qualification import cuda_v14_core_batch as core
from research.g6.cuda_qualification import v14_core_batch_contract as contract
validation = json.loads((CONTROL_INPUT / 'v14_core_validation.json').read_text(encoding='utf-8'))
identity = contract.validate_core_validation_record(validation)
print('Qualified controls and full-domain validation: PASS')
print('Core validation identity:', core.identity_sha256(identity))
print('Preparation output:', OUT)

In [ ]:
# Non-value only: the core producer path reconstructs and discards records for this timing sample.
ENV = dict(os.environ, PYTHONDONTWRITEBYTECODE='1')
def run_json(name, arguments):
    destination = OUT / name
    with destination.open('w', encoding='utf-8') as stream:
        result = subprocess.run(arguments, cwd=ROOT, env=ENV, stdout=stream, stderr=subprocess.PIPE, text=True)
    if result.returncode:
        raise RuntimeError('Command failed: ' + ' '.join(arguments) + '\n\n' + result.stderr[-8000:])
    return json.loads(destination.read_text(encoding='utf-8'))

measurement_run = run_json('v14_core_measurement_0_5x_0_4096.json', [
    'python3', 'research/g6/run_cuda_v14_core_batches.py', '--measure',
    '--scale', '0.5x', '--start', '0', '--stop', '4096',
    '--unbalanced-record', str(CONTROL_INPUT / 'unbalanced_qualification.json'),
    '--slack-record', str(CONTROL_INPUT / 'slack_qualification.json'),
])
assert measurement_run['verdict'] == 'measured', measurement_run
measurement = measurement_run['throughput_measurement']
(OUT / 'v14_core_throughput_measurement_0_5x.json').write_text(
    json.dumps(measurement, sort_keys=True, separators=(',', ':')), encoding='utf-8'
)
sample = measurement['samples'][0]
print('0.5x measurement: PASS')
print('coordinates:', sample['coordinates'])
print('wall seconds:', sample['wall_seconds'])
print('coordinate seconds:', sample['coordinate_seconds'])
print('setup seconds:', sample['setup_seconds'])

In [ ]:
plan = run_json('v14_0_5x_core_batch_plan.json', [
    'python3', 'research/g6/plan_v14_cuda_core_batches.py', 'plan', '--scale', '0.5x',
    '--validation-record', str(CONTROL_INPUT / 'v14_core_validation.json'),
    '--measurement', str(OUT / 'v14_core_throughput_measurement_0_5x.json'),
])
assert plan['scale'] == '0.5x' and plan['content_sha256']
sizing = plan['sizing']
print('0.5x plan: PASS')
print('acceptance:', sizing['measurement_acceptance_basis'])
print('coordinates per batch:', sizing['coordinates_per_batch'])
print('batch count:', sizing['batch_count'])
print('plan content SHA-256:', plan['content_sha256'])
print('Stop here. This notebook does not produce 0.5x records.')